# In-vitro Training Oracles

This notebook trains LNPHub in-vitro expression regression oracles from four model families. The heavy code lives in reusable Python modules under `src/lnphub_oracles/`; the notebook only launches each training group and shows the saved manifests.

Outputs are written to `/content/lnphub-submission/trained_oracles/in_vitro_expression/` with one folder per oracle.

## 0. Clone Private Repository

This clones the private repository from the `dev` branch. Use a fine-grained GitHub token with access to `Maburidi/lnphub-submission`.

In [ ]:
# Cell 1 - Clone private LNPHub repo from dev branch
from pathlib import Path
from getpass import getpass
import os
import subprocess

REPO_URL = "https://github.com/Maburidi/lnphub-submission.git"
BRANCH = "dev"
PROJECT_DIR = Path("/content/lnphub-submission")


def get_github_token():
    token = os.environ.get("GITHUB_TOKEN", "").strip()
    if token:
        return token
    try:
        from google.colab import userdata
        token = (userdata.get("GITHUB_TOKEN") or "").strip()
    except Exception:
        token = ""
    return token or getpass("GitHub token for the private repo: ").strip()


def run(cmd, token=None):
    printable = " ".join(map(str, cmd))
    if token:
        printable = printable.replace(token, "***TOKEN***")
    print("$", printable)
    res = subprocess.run(cmd, text=True, capture_output=True)
    out = (res.stdout + res.stderr).replace(token or "", "***TOKEN***")
    if out.strip():
        print(out[-4000:])
    if res.returncode != 0:
        raise RuntimeError(f"Command failed with code {res.returncode}")
    return res


token = get_github_token()
auth_repo_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")

if PROJECT_DIR.exists():
    print(f"Repo already exists: {PROJECT_DIR}")
    run(["git", "-C", str(PROJECT_DIR), "remote", "set-url", "origin", auth_repo_url], token=token)
    run(["git", "-C", str(PROJECT_DIR), "fetch", "origin", BRANCH], token=token)
    run(["git", "-C", str(PROJECT_DIR), "checkout", BRANCH], token=token)
    run(["git", "-C", str(PROJECT_DIR), "pull", "origin", BRANCH], token=token)
    run(["git", "-C", str(PROJECT_DIR), "remote", "set-url", "origin", REPO_URL], token=token)
else:
    run(["git", "clone", "--branch", BRANCH, "--single-branch", auth_repo_url, str(PROJECT_DIR)], token=token)
    run(["git", "-C", str(PROJECT_DIR), "remote", "set-url", "origin", REPO_URL], token=token)

os.chdir(PROJECT_DIR)
print("Working directory:", Path.cwd())
run(["git", "-C", str(PROJECT_DIR), "branch", "--show-current"])

## 1. Install Dependencies and Define Paths

In [ ]:
# Cell 2 - Install oracle-training dependencies and expose local src package
import os
import sys
import subprocess
from pathlib import Path

PROJECT_DIR = Path("/content/lnphub-submission")
SRC_DIR = PROJECT_DIR / "src"
REQ_PATH = PROJECT_DIR / "requirements-oracles.txt"

if not PROJECT_DIR.exists():
    raise FileNotFoundError(f"Repo was not cloned/found: {PROJECT_DIR}")
if not SRC_DIR.exists():
    raise FileNotFoundError(
        f"Missing {SRC_DIR}. The cloned dev branch does not contain the oracle-training scripts yet. "
        "Pull the latest dev branch first."
    )

if REQ_PATH.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REQ_PATH)], check=False)
else:
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "numpy", "pandas", "pyarrow", "scikit-learn", "scipy", "joblib",
        "matplotlib", "seaborn", "tqdm", "xgboost", "rdkit",
        "torch", "torch-geometric", "transformers", "accelerate", "safetensors",
    ], check=False)

# Colab/Python versions can disagree on RDKit package names, so verify and repair explicitly.
try:
    import rdkit
except ModuleNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rdkit"], check=False)
    try:
        import rdkit
    except ModuleNotFoundError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rdkit-pypi"], check=False)
        import rdkit

os.environ["PYTHONPATH"] = str(SRC_DIR) + os.pathsep + os.environ.get("PYTHONPATH", "")
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import lnphub_oracles
print("Oracle package loaded from:", lnphub_oracles.__file__)
print("RDKit version:", rdkit.__version__)
print("PYTHONPATH begins with:", os.environ["PYTHONPATH"].split(os.pathsep)[0])


In [ ]:
# Cell 3 - Training paths and input checks
from pathlib import Path
import pandas as pd
from IPython.display import display

PROJECT_DIR = Path("/content/lnphub-submission")
SUBSET_PATH = PROJECT_DIR / "data_subsets" / "lnphub_oracle_in_vitro_expression_gt400.csv"
ENCODED_DIR = PROJECT_DIR / "encoded_in_vitro"
SPLIT_DIR = PROJECT_DIR / "oracle_splits" / "in_vitro_expression"
TRAINVAL_PATH = SPLIT_DIR / "lnphub_oracle_in_vitro_expression_trainval_5fold.csv"
UNTOUCHED_TEST_PATH = SPLIT_DIR / "lnphub_oracle_in_vitro_expression_untouched_test.csv"
ENCODED_TRAINVAL_DIR = SPLIT_DIR / "encoded_trainval"
ENCODED_TEST_DIR = SPLIT_DIR / "encoded_untouched_test"
OUTPUT_DIR = PROJECT_DIR / "trained_oracles" / "in_vitro_expression"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not SUBSET_PATH.exists():
    raise FileNotFoundError(SUBSET_PATH)
if not ENCODED_DIR.exists():
    raise FileNotFoundError(ENCODED_DIR)

encoded_files = sorted(ENCODED_DIR.glob("*_encoder_vectors.parquet"))
print("Full subset:", SUBSET_PATH)
print("Encoded vector folder:", ENCODED_DIR)
print("Split output folder:", SPLIT_DIR)
print("Model output folder:", OUTPUT_DIR)
print("Encoded files:")
for path in encoded_files:
    print("-", path.name)

subset = pd.read_csv(SUBSET_PATH, low_memory=False)
print("Full subset shape:", subset.shape)
display(subset[[c for c in ["record_id", "study_id", "lnp_id", "il_id", "delivery", "delivery_metric", "experimental_setting"] if c in subset.columns]].head())


## 2. Create Untouched Test Set and 5-Fold CV Splits

This creates a deterministic held-out test set that is never used for model fitting or cross-validation. Five CV folds are assigned only inside the remaining trainval set. Matching encoded-vector files are saved for both trainval and untouched test.

In [ ]:
# Cell 4 - Save untouched test set, trainval set, CV folds, and matching encoded vectors
import sys
import subprocess

cmd = [
    sys.executable, "-m", "lnphub_oracles.prepare_splits",
    "--subset", str(SUBSET_PATH),
    "--encoded-dir", str(ENCODED_DIR),
    "--output-dir", str(SPLIT_DIR),
    "--split-name", "in_vitro_expression",
    "--id-col", "record_id",
    "--group-col", "study_id",
    "--target-col", "delivery",
    "--test-size", "0.15",
    "--test-groups", "LNPhub_JL_2024", "LNPhub_BL_2024",
    "--n-folds", "5",
    "--seed", "7",
]
subprocess.run(cmd, check=True)

In [ ]:
# Cell 5 - Inspect saved split files
import pandas as pd
from IPython.display import display

trainval = pd.read_csv(TRAINVAL_PATH, low_memory=False)
untouched_test = pd.read_csv(UNTOUCHED_TEST_PATH, low_memory=False)
encoded_summary = pd.read_csv(SPLIT_DIR / "lnphub_oracle_in_vitro_expression_encoded_split_summary.csv")

print("Trainval:", TRAINVAL_PATH, trainval.shape)
print("Untouched test:", UNTOUCHED_TEST_PATH, untouched_test.shape)
print("Encoded trainval folder:", ENCODED_TRAINVAL_DIR)
print("Encoded untouched-test folder:", ENCODED_TEST_DIR)

print("\nTrainval CV fold sizes:")
display(trainval["cv_fold"].value_counts().sort_index().rename_axis("cv_fold").reset_index(name="rows"))
print("\nUntouched test studies:")
display(untouched_test["study_id"].value_counts().rename_axis("study_id").reset_index(name="rows"))
print("\nEncoded vector split summary:")
display(encoded_summary)

## 3. Molecular Fingerprints and RDKit Descriptors

Methods: ECFP4 + MLP, ECFP4 + XGBoost, ECFP4 + Random Forest, RDKit Descriptors + MLP, RDKit Descriptors + XGBoost, RDKit Descriptors + Random Forest.

In [ ]:
# Cell 6 - Train molecular fingerprint / descriptor oracles
import os
import sys
import subprocess
from pathlib import Path

PROJECT_DIR = Path("/content/lnphub-submission")
SRC_DIR = PROJECT_DIR / "src"
env = os.environ.copy()
env["PYTHONPATH"] = str(SRC_DIR) + os.pathsep + env.get("PYTHONPATH", "")

cmd = [
    sys.executable, "-m", "lnphub_oracles.train_tabular",
    "--subset", str(TRAINVAL_PATH),
    "--test-subset", str(UNTOUCHED_TEST_PATH),
    "--output-dir", str(OUTPUT_DIR),
    "--target-col", "delivery",
    "--methods",
    "ecfp4_mlp",
    "ecfp4_xgboost",
    "ecfp4_random_forest",
    "rdkit_descriptors_mlp",
    "rdkit_descriptors_xgboost",
    "rdkit_descriptors_random_forest",
]

res = subprocess.run(cmd, text=True, capture_output=True, env=env)
print("STDOUT:")
print(res.stdout)
print("\nSTDERR:")
print(res.stderr)
if res.returncode != 0:
    raise RuntimeError(f"Tabular training failed with exit code {res.returncode}")


## 4. Graph Neural Networks

Methods: GCN, GIN, D-MPNN. These train directly from ionizable-lipid SMILES graphs.

In [ ]:
# Cell 5 - Train graph neural network oracles
import sys
import subprocess

cmd = [
    sys.executable, "-m", "lnphub_oracles.train_gnn",
    "--subset", str(TRAINVAL_PATH),
    "--test-subset", str(UNTOUCHED_TEST_PATH),
    "--output-dir", str(OUTPUT_DIR),
    "--target-col", "delivery",
    "--methods", "gcn", "gin", "dmpnn",
    "--epochs", "120",
    "--batch-size", "64",
]
subprocess.run(cmd, check=True)

## 5. Molecular Pretrained Models - Frozen

Methods: AGILE-Frozen, KPGT-Frozen, LiON-Frozen, ChemBERTa-MTR-Frozen, MoLFormer-Frozen, MolCLR-GIN-Frozen, Uni-Mol-Frozen.

Each method trains a supervised MLP head on already-saved frozen encoder vectors.

In [ ]:
# Cell 6 - Train frozen pretrained-embedding oracles
import sys
import subprocess

cmd = [
    sys.executable, "-m", "lnphub_oracles.train_frozen",
    "--encoded-dir", str(ENCODED_TRAINVAL_DIR),
    "--test-encoded-dir", str(ENCODED_TEST_DIR),
    "--output-dir", str(OUTPUT_DIR),
    "--target-col", "delivery",
    "--estimator", "mlp",
    "--encoders",
    "AGILE", "KPGT", "LiON", "ChemBERTa_MTR", "MoLFormer", "MolCLR_GIN", "Uni_Mol",
]
subprocess.run(cmd, check=True)

## 6. Molecular Pretrained Models - Fine-Tuned

Methods requested: AGILE-Finetuned, KPGT-Finetuned, LiON-Finetuned, ChemBERTa-MTR-Finetuned, MoLFormer-Finetuned, MolCLR-GIN-Finetuned, Uni-Mol-Finetuned.

This cell trains the fine-tuning adapters that are directly runnable from this repository/runtime. Methods requiring missing official checkpoints or unavailable official fine-tuning interfaces are recorded in the manifest as `not_trained` with the reason.

In [ ]:
# Cell 7 - Fine-tune supported pretrained encoders
import sys
import subprocess

cmd = [
    sys.executable, "-m", "lnphub_oracles.train_finetuned",
    "--subset", str(TRAINVAL_PATH),
    "--test-subset", str(UNTOUCHED_TEST_PATH),
    "--output-dir", str(OUTPUT_DIR),
    "--repo-dir", "/content/notebooks/encoder_repos",
    "--target-col", "delivery",
    "--methods",
    "agile", "kpgt", "lion", "chemberta_mtr", "molformer", "molclr_gin", "unimol",
    "--epochs", "8",
    "--batch-size", "32",
]
subprocess.run(cmd, check=True)

## 7. Training Summary

In [ ]:
# Cell 8 - Collect all training manifests
from pathlib import Path
import pandas as pd
from IPython.display import display

manifest_files = sorted(OUTPUT_DIR.glob("*_training_manifest.csv"))
if not manifest_files:
    raise FileNotFoundError(f"No training manifests found in {OUTPUT_DIR}")

all_manifests = []
for path in manifest_files:
    tmp = pd.read_csv(path)
    tmp.insert(0, "manifest", path.name)
    all_manifests.append(tmp)

summary = pd.concat(all_manifests, ignore_index=True)
summary.to_csv(OUTPUT_DIR / "all_training_manifests.csv", index=False)
display(summary)

print("Saved combined manifest:", OUTPUT_DIR / "all_training_manifests.csv")
print("Oracle folders:")
for p in sorted([x for x in OUTPUT_DIR.iterdir() if x.is_dir()]):
    print("-", p)

In [ ]:
# Cell 9 - Optional: zip trained in-vitro oracles for download
# Uncomment the last two lines when you want to download the trained models from Colab.

import shutil
from pathlib import Path

zip_base = "/content/lnphub_in_vitro_expression_trained_oracles"
shutil.make_archive(zip_base, "zip", OUTPUT_DIR)
print(f"Created: {zip_base}.zip")

# from google.colab import files
# files.download(f"{zip_base}.zip")